In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import chi2_contingency, kruskal

In [2]:
FILE_PATH = Path(
    "../data/raw/張瓊之_給馬老師資料20250925_V4(遺失值不用登錄成-1).xlsx"
)

demo = pd.read_excel(FILE_PATH, sheet_name="demographic")
mmse = pd.read_excel(FILE_PATH, sheet_name="MMSE longitudinal")
casi = pd.read_excel(FILE_PATH, sheet_name="CASI longitudinal")

print("demographic shape:", demo.shape)
print("MMSE shape:", mmse.shape)
print("CASI shape:", casi.shape)

demographic shape: (1302, 10)
MMSE shape: (5550, 11)
CASI shape: (4353, 12)


In [3]:
mmse = mmse.rename(columns={
    "Count Number": "patient_id",
    "Date": "visit_date",
    "MMSE(numerical)": "mmse",
    "CDR (scale)": "cdr_global",
    "CDR_M (scale)": "cdr_memory",
    "CDR_O (scale)": "cdr_orientation",
    "CDR_J (scale)": "cdr_judgment",
    "CDR_C (scale)": "cdr_community",
    "CDR_H (scale)": "cdr_home_hobbies",
    "CDR_P (scale)": "cdr_personal_care",
    "CDR-SOB(numerical)": "cdr_sob"
})

mmse.head()

,patient_id,visit_date,mmse,cdr_global,cdr_memory,cdr_orientation,cdr_judgment,cdr_community,cdr_home_hobbies,cdr_personal_care,cdr_sob
0,1,2006-11-13,17,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0
1,1,2007-08-31,15,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0
2,1,2008-01-31,15,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0
3,1,2008-08-14,16,1.0,1.0,1.0,1.0,1.0,1.0,0.0,5.0
4,1,2009-02-05,17,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0


In [16]:
mmse["patient_id"].unique()
set1 = set([i for i in range(1,1302)])
output = set1 - set(mmse["patient_id"])
print("mmse資料表缺失的病人id:", output)

mmse資料表缺失的病人id: {970, 458, 629, 694}


In [4]:
casi = casi.rename(columns={
    "Count Number": "patient_id",
    "Date": "visit_date",
    "MENMA10": "casi_mental_manipulation",
    "ATTEN8": "casi_attention",
    "ORIEN18": "casi_orientation",
    "LTM10": "casi_long_term_memory",
    "STM12": "casi_short_term_memory",
    "ABSTR12": "casi_abstraction",
    "DRAW10": "casi_drawing",
    "ANML10": "casi_verbal_fluency",
    "LANG10": "casi_language",
    "Total": "casi_total"
})

casi.head()

,patient_id,visit_date,casi_mental_manipulation,casi_attention,casi_orientation,casi_long_term_memory,casi_short_term_memory,casi_abstraction,casi_drawing,casi_verbal_fluency,casi_language,casi_total
0,1,2008-08-14,5,6,5,8,1.8,8,8,2,5.0,48.8
1,1,2009-02-05,1,7,7,6,2.8,7,10,0,9.0,49.3
2,1,2009-08-24,2,5,2,6,1.8,6,0,3,8.0,33.3
3,1,2010-02-04,6,5,8,10,3.0,5,4,2,8.0,51.0
4,1,2010-08-06,2,5,3,6,0.0,6,4,1,4.0,31.0


In [5]:
mmse_casi = mmse.merge(
    casi,
    on=["patient_id", "visit_date"],
    how="left"
)

mmse_casi.head()

,patient_id,visit_date,mmse,cdr_global,cdr_memory,cdr_orientation,cdr_judgment,cdr_community,cdr_home_hobbies,cdr_personal_care,...,casi_mental_manipulation,casi_attention,casi_orientation,casi_long_term_memory,casi_short_term_memory,casi_abstraction,casi_drawing,casi_verbal_fluency,casi_language,casi_total
0,1,2006-11-13,17,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2007-08-31,15,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,2008-01-31,15,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,2008-08-14,16,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,5.0,6.0,5.0,8.0,1.8,8.0,8.0,2.0,5.0,48.8
4,1,2009-02-05,17,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,7.0,7.0,6.0,2.8,7.0,10.0,0.0,9.0,49.3


In [6]:
import pandas as pd
from scipy.stats import ttest_ind, mannwhitneyu

df = mmse_casi.copy()

df["visit_date"] = pd.to_datetime(df["visit_date"])
df["mmse"] = pd.to_numeric(df["mmse"], errors="coerce")
df["casi"] = pd.to_numeric(df["casi_total"], errors="coerce")

# 判定每位病人的 CASI 是否完全缺失
casi_status = (
    df.groupby("patient_id")["casi"]
      .apply(lambda x: x.notna().any())
      .rename("has_any_casi")
      .reset_index()
)

# 加回原始資料
df = df.merge(casi_status, on="patient_id", how="left")

df["casi_complete_missing"] = (~df["has_any_casi"]).astype(int)

# 每位病人的第一筆有效 MMSE
baseline_mmse = (
    df.dropna(subset=["mmse"])
      .sort_values(["patient_id", "visit_date"])
      .groupby("patient_id", as_index=False)
      .first()
)

baseline_mmse.groupby("casi_complete_missing")["mmse"].agg(
    ["count", "mean", "std", "median", "min", "max"]
)

,count,mean,std,median,min,max
casi_complete_missing,,,,,,
0,914,22.018600,6.088853,23.0,0,30
1,383,23.060052,7.040202,26.0,0,30


In [17]:
383/ 1297

0.2952968388589052

In [7]:
df_has_casi = df[df["casi_complete_missing"] == 1]
df_has_casi["patient_id"].unique()

array([   6,   95,  191,  400,  442,  463,  465,  466,  501,  502,  503,
        504,  505,  506,  507,  508,  509,  510,  511,  512,  513,  514,
        515,  516,  517,  518,  519,  520,  521,  522,  523,  524,  525,
        526,  644,  645,  646,  647,  648,  649,  650,  651,  652,  653,
        654,  655,  656,  657,  658,  659,  660,  661,  662,  663,  664,
        665,  666,  667,  668,  669,  670,  671,  672,  673,  674,  675,
        676,  677,  678,  679,  701,  702,  704,  705,  706,  707,  708,
        709,  710,  711,  712,  713,  714,  715,  716,  717,  718,  719,
        720,  722,  729,  796,  816,  849,  850,  851,  852,  853,  854,
        855,  856,  857,  858,  859,  860,  861,  862,  863,  864,  865,
        866,  867,  868,  869,  870,  871,  872,  873,  874,  875,  876,
        877,  878,  879,  880,  881,  882,  883,  884,  885,  886,  887,
        888,  889,  890,  891,  892,  893,  894,  895,  896,  897,  898,
        899,  900,  901,  902,  903,  914,  915,  9

In [8]:
group_missing = baseline_mmse.loc[
    baseline_mmse["casi_complete_missing"] == 1, "mmse"
].dropna()

group_other = baseline_mmse.loc[
    baseline_mmse["casi_complete_missing"] == 0, "mmse"
].dropna()

t_stat, p_value = ttest_ind(
    group_missing,
    group_other,
    equal_var=False
)

print(f"CASI 完全缺失組 n = {len(group_missing)}")
print(f"其餘病人組 n = {len(group_other)}")
print(f"Welch's t = {t_stat:.4f}")
print(f"p-value = {p_value:.4g}")

CASI 完全缺失組 n = 383
其餘病人組 n = 914
Welch's t = 2.5261
p-value = 0.01178
